# T2I-RL Full Training — GRPO + Composite Reward

**正式训练版本** — 使用 1029 条 prompts，CLIP + SiliconFlow VLM 复合 Reward

| 参数 | 值 |
|------|----|
| 训练数据 | `train_prompts_large.json` — 1029 prompts (13 类别，核心类 ~100条) |
| Reward | CLIP (0.3) + SiliconFlow Qwen2.5-VL-32B (0.7) |
| 算法 | GRPO (group_size=4) |
| LoRA | r=16, alpha=32 |
| Epochs | 5 |
| 预计时长 | ~12-14 小时 (A100) / ~28 小时 (T4) |
| API 费用 | ~¥8-10 (Qwen2.5-VL-32B, 5 epochs) |

> **与旧版 `colab_siliconflow_training.ipynb` 的区别**  
> 旧版为快速验证 (50 prompts, 3 epochs, VLM only)。  
> 本 notebook 为正式训练，结果用于最终报告。

---

## Step 0: 训练配置

**只需修改这一个 cell，其余全部自动运行。**

In [ ]:
import os

# ============================================================
# [必填] SiliconFlow API Key
# 获取地址: https://cloud.siliconflow.cn/
# ============================================================
SILICONFLOW_API_KEY = ""  # <-- 填入你的 key

# ============================================================
# [可选] W&B 实验跟踪
# 留空则不使用 W&B
# ============================================================
WANDB_API_KEY = ""  # <-- 填入你的 W&B key（可留空）

# ============================================================
# VLM 模型选择
# - "Qwen/Qwen2.5-VL-32B-Instruct"  推荐，~¥15-20 总花费
# - "THUDM/GLM-4.1V-9B-Thinking"    免费，效果略差
# ============================================================
VLM_MODEL = "Qwen/Qwen2.5-VL-32B-Instruct"

# ============================================================
# 训练超参数
# 时间估算 (1029 prompts, group_size=4, ~8s/step on A100):
#   5 epochs  → ~12-14 小时 on A100  (推荐)
#   3 epochs  → ~7-8  小时 on A100  (快速验证)
#   10 epochs → ~23-28 小时 on A100 (不推荐，超出 Colab 连接时限)
# ============================================================
NUM_EPOCHS       = 5
LORA_RANK        = 16
GROUP_SIZE       = 4     # GRPO: 每个 prompt 生成几张图
LEARNING_RATE    = 1e-5
GRAD_ACCUM_STEPS = 8     # effective batch size = 1 × 8 = 8
SAVE_STEPS       = 200   # 每隔多少 step 保存一次
LOG_STEPS        = 10

# These defaults are auto-adjusted by Step 3 based on GPU VRAM.
# You can override them here if you know your GPU.
KL_COEF          = 0.01   # auto-set to 0 on T4 (< 20 GB VRAM)
PPO_EPOCHS       = 3      # auto-set to 2 on T4

# ============================================================
# Google Drive 持久化（推荐开启，防止 Colab 断连丢失进度）
# ============================================================
USE_GOOGLE_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/T2I-RL/outputs/grpo_full"
LOCAL_OUTPUT_DIR = "./outputs/grpo_full"

# ============================================================
# 输出目录（自动选择 Drive 或本地）
# ============================================================
OUTPUT_DIR = DRIVE_OUTPUT_DIR if USE_GOOGLE_DRIVE else LOCAL_OUTPUT_DIR

# 写入环境变量
os.environ["SILICONFLOW_API_KEY"] = SILICONFLOW_API_KEY
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY

print("Configuration:")
print(f"  VLM model    : {VLM_MODEL}")
print(f"  Epochs       : {NUM_EPOCHS}")
print(f"  LoRA rank    : {LORA_RANK}")
print(f"  Group size   : {GROUP_SIZE}")
print(f"  Output dir   : {OUTPUT_DIR}")
print(f"  W&B logging  : {'enabled' if WANDB_API_KEY else 'disabled'}")
print(f"  Google Drive : {'enabled' if USE_GOOGLE_DRIVE else 'disabled'}")

if not SILICONFLOW_API_KEY:
    print("\nWARNING: SILICONFLOW_API_KEY is empty! VLM reward will fail.")

## Step 1: 挂载 Google Drive

挂载后 checkpoint 会自动同步到云端，断连重连后可继续训练。

In [ ]:
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print(f"Google Drive mounted. Output will be saved to: {DRIVE_OUTPUT_DIR}")
else:
    os.makedirs(LOCAL_OUTPUT_DIR, exist_ok=True)
    print(f"Using local storage: {LOCAL_OUTPUT_DIR}")

## Step 2: 安装依赖

In [ ]:
import time
start = time.time()

!pip install -q torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.49.0 accelerate safetensors
!pip install -q --no-deps git+https://github.com/deepseek-ai/Janus.git
!pip install -q attrdict sentencepiece
!pip install -q open-clip-torch timm einops peft bitsandbytes
!pip install -q tqdm Pillow matplotlib
!pip install -q openai
!pip install -q -U wandb  # always install (upgraded) — login is optional

print(f"\nInstallation complete: {time.time()-start:.1f}s")

## Step 3: 检查 GPU

In [ ]:
import torch, os

# Reduce CUDA memory fragmentation
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU      : {gpu_name}")
    print(f"VRAM     : {gpu_mem:.1f} GB")

    # ===== VRAM-based auto-configuration =====
    if gpu_mem < 20:
        # T4 (15 GB) or similar — aggressive memory savings
        GROUP_SIZE = min(GROUP_SIZE, 2)
        KL_COEF = 0.0       # no reference model to save ~2-5 GB
        PPO_EPOCHS = 2      # fewer replay passes
        print(f"\n[Auto-config] Low VRAM ({gpu_mem:.1f} GB):")
        print(f"  GROUP_SIZE  -> {GROUP_SIZE}")
        print(f"  kl_coef     -> {KL_COEF} (KL disabled, no reference model)")
        print(f"  ppo_epochs  -> {PPO_EPOCHS}")
    elif gpu_mem < 30:
        # L4 (24 GB) — moderate
        KL_COEF = 0.01
        PPO_EPOCHS = 3
        print(f"\n[Auto-config] Medium VRAM ({gpu_mem:.1f} GB):")
        print(f"  GROUP_SIZE  -> {GROUP_SIZE}")
        print(f"  kl_coef     -> {KL_COEF}")
        print(f"  ppo_epochs  -> {PPO_EPOCHS}")
    else:
        # A100 (40/80 GB) — full settings
        KL_COEF = 0.01
        PPO_EPOCHS = 3
        print(f"\n[Auto-config] High VRAM ({gpu_mem:.1f} GB): using default settings.")
else:
    raise RuntimeError("No GPU found. Please enable GPU runtime in Colab: Runtime > Change runtime type > GPU")

## Step 4: 克隆 / 更新项目代码

In [ ]:
if not os.path.exists('T2I-RL-Project'):
    !git clone https://github.com/Inoriany/T2I-RL-Project.git
    print("Repository cloned")
else:
    !cd T2I-RL-Project && git pull
    print("Repository updated")

os.chdir('T2I-RL-Project')

import sys
sys.path.insert(0, os.path.abspath('src'))  # for: from models.xxx import
sys.path.insert(0, os.path.abspath('.'))    # for: from src.xxx import (internal)

print(f"Working directory: {os.getcwd()}")
print(f"Train data: {os.path.exists('data/prompts/train_prompts_large.json')}")

## Step 5: 测试 SiliconFlow API

In [ ]:
from openai import OpenAI
import base64, io
from PIL import Image

test_img = Image.new('RGB', (64, 64), color=(200, 100, 50))
buf = io.BytesIO()
test_img.save(buf, format='PNG')
img_b64 = base64.b64encode(buf.getvalue()).decode()

client = OpenAI(
    api_key=os.environ["SILICONFLOW_API_KEY"],
    base_url="https://api.siliconflow.cn/v1"
)

resp = client.chat.completions.create(
    model=VLM_MODEL,
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": "What color is this image? Reply in one word."},
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_b64}"}},
        ]
    }],
    max_tokens=20,
)

print(f"SiliconFlow API test passed")
print(f"Model     : {VLM_MODEL}")
print(f"Response  : {resp.choices[0].message.content}")

## Step 6: 加载 Janus-Pro-1B + LoRA

In [ ]:
from models.generators import JanusProGenerator

print("Loading Janus-Pro-1B (4-bit quantization)...")
generator = JanusProGenerator(
    model_name_or_path="deepseek-ai/Janus-Pro-1B",
    device="cuda",
    load_in_4bit=True,
)
generator.load_model()

# Enable LoRA
if not getattr(generator, 'lora_enabled', False):
    generator.enable_lora(lora_config={
        'r': LORA_RANK,
        'lora_alpha': LORA_RANK * 2,
        'lora_dropout': 0.05,
        'target_modules': ['q_proj', 'v_proj', 'k_proj', 'o_proj',
                           'gate_proj', 'up_proj', 'down_proj'],
    })
    print(f"LoRA enabled (rank={LORA_RANK})")

trainable = sum(p.numel() for p in generator.model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in generator.model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

# Quick sanity check
torch.cuda.empty_cache()
test_img = generator.generate(prompt="a red apple")[0]
print("Generator test passed")
display(test_img)

## Step 7: 创建 Composite Reward (CLIP + SiliconFlow VLM)

In [ ]:
from models.reward_models import CLIPRewardModel, VLMRewardModel, CompositeRewardModel

# CLIP reward (本地运行，无 API 费用)
clip_reward = CLIPRewardModel(
    device="cuda",
    model_name="ViT-L-14",
    pretrained="openai",
)
clip_reward.load_model()
print("CLIP reward model loaded (ViT-L-14)")

# VLM reward (SiliconFlow API)
vlm_reward = VLMRewardModel(
    device="cuda",
    use_api=True,
    api_model=VLM_MODEL,
)
vlm_reward.load_model()
print(f"VLM reward model ready ({VLM_MODEL})")

# Composite: CLIP gate (0.3) + VLM (0.7)
reward_model = CompositeRewardModel(
    reward_models={"clip": clip_reward, "vlm": vlm_reward},  # must be dict, not list,
    weights={"clip": 0.3, "vlm": 0.7},
)

# Test composite reward
torch.cuda.empty_cache()
test_out = reward_model.compute_reward([test_img], ["a red apple"])
print(f"Composite reward test passed: {test_out.rewards.item():.4f}")

## Step 8: 加载训练数据 (1029 prompts)

In [ ]:
import json, random
from data.dataset import PromptDataset
from torch.utils.data import DataLoader

def load_prompts(filepath):
    with open(filepath, 'r') as f:
        data = json.load(f)
    prompts_raw = data.get('prompts', data)
    if isinstance(prompts_raw, dict):
        flat = []
        for items in prompts_raw.values():
            if isinstance(items, list):
                flat.extend([x if isinstance(x, str) else x['prompt'] for x in items])
        return flat
    if isinstance(prompts_raw, list):
        return [x if isinstance(x, str) else x['prompt'] for x in prompts_raw]
    raise ValueError(f"Unsupported prompt format in {filepath}")

# Load full dataset
train_prompts = load_prompts('data/prompts/train_prompts_large.json')
val_prompts   = load_prompts('data/prompts/val_prompts.json')

random.shuffle(train_prompts)

print(f"Train prompts : {len(train_prompts)}")
print(f"Val prompts   : {len(val_prompts)}")
print(f"\nSample prompts:")
for p in train_prompts[:5]:
    print(f"  - {p}")

train_dataset = PromptDataset(train_prompts)
val_dataset   = PromptDataset(val_prompts)
collate_fn = lambda x: {'prompt': [item['prompt'] for item in x]}

train_dataloader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn,
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn,
)

print(f"\nTrain dataloader: {len(train_dataloader)} batches/epoch")
print(f"Val dataloader  : {len(val_dataloader)} batches/eval")

## Step 9: 配置 GRPO Trainer

In [ ]:
from training.grpo_trainer import GRPOTrainer, GRPOConfig

config = GRPOConfig(
    num_epochs=NUM_EPOCHS,
    batch_size=1,
    learning_rate=LEARNING_RATE,
    num_samples_per_prompt=GROUP_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    kl_coef=KL_COEF,
    clip_ratio=0.2,
    ppo_epochs=PPO_EPOCHS,
    eval_steps=SAVE_STEPS,
    save_steps=SAVE_STEPS,
    logging_steps=LOG_STEPS,
    output_dir=OUTPUT_DIR,
    use_wandb=False,  # W&B is initialized manually in Step 11
)

trainer = GRPOTrainer(
    generator=generator,
    reward_model=reward_model,
    train_dataloader=train_dataloader,
    eval_dataloader=val_dataloader,
    grpo_config=config,
)

print("GRPO Trainer configured")
print(f"  Epochs           : {NUM_EPOCHS}")
print(f"  Prompts/epoch    : {len(train_dataloader)}")
print(f"  Images/step      : {GROUP_SIZE}")
print(f"  Effective batch  : {GROUP_SIZE * GRAD_ACCUM_STEPS}")
print(f"  Est. total steps : ~{len(train_dataloader) * NUM_EPOCHS}")
print(f"  Save every       : {SAVE_STEPS} steps")
print(f"  Eval every       : {SAVE_STEPS} steps")
print(f"  Clip ratio       : {config.clip_ratio}")
print(f"  PPO epochs       : {PPO_EPOCHS}")
print(f"  KL coef          : {KL_COEF}")
print(f"  Output dir       : {OUTPUT_DIR}")
print("\nWatch these metrics in logs/W&B:")
print("  - train/reward_mean")
print("  - train/reward_clip_mean")
print("  - train/reward_vlm_mean")
print("  - train/vlm_parse_error_rate")
print("  - train/ratio_mean")
print("  - train/clip_fraction")
print("  - train/kl_div")
print("  - eval/avg_reward")

## Step 10: 从 Checkpoint 恢复（如果有）

Colab 断连后重新运行到这里，会自动从上次保存的 checkpoint 继续。

In [ ]:
import glob

# 寻找最新的 checkpoint
crash_ckpt = os.path.join(OUTPUT_DIR, 'crash_checkpoint', 'training_state.pt')
step_ckpts = sorted(
    glob.glob(os.path.join(OUTPUT_DIR, 'checkpoint-*', 'training_state.pt')),
    key=os.path.getmtime
)

resume_path = None
if os.path.exists(crash_ckpt):
    resume_path = os.path.dirname(crash_ckpt)
    print(f"Found crash checkpoint: {resume_path}")
elif step_ckpts:
    resume_path = os.path.dirname(step_ckpts[-1])
    print(f"Found step checkpoint: {resume_path}")
else:
    print("No checkpoint found, starting from scratch.")

if resume_path:
    trainer.load_checkpoint(resume_path)
    trainable = [p for p in trainer.generator.model.parameters() if p.requires_grad]
    if not trainable:
        raise RuntimeError("No trainable params after resume. Re-run Steps 6-10 from scratch.")
    print(f"Resumed from step {trainer.global_step}, epoch {trainer.current_epoch}")

## Step 11: 开始训练

预计时长 ~12-14 小时 (A100, 5 epochs)。
- Checkpoint 每 **200 steps** 自动保存到 Google Drive
- 如果 Colab 断连，重连后从 **Step 4** 开始重新运行即可自动恢复

In [ ]:
from datetime import datetime

# ---------- W&B (non-blocking) ----------
USE_WANDB = False
if WANDB_API_KEY:
    try:
        import wandb
        wandb.login(key=WANDB_API_KEY)
        wandb.init(
            project="t2i-rl",
            name=f"grpo-full-{datetime.now().strftime('%Y%m%d-%H%M%S')}",
            config={
                "num_epochs": NUM_EPOCHS,
                "lora_rank": LORA_RANK,
                "group_size": GROUP_SIZE,
                "learning_rate": LEARNING_RATE,
                "clip_ratio": config.clip_ratio,
                "ppo_epochs": config.ppo_epochs,
                "kl_coef": config.kl_coef,
                "num_prompts": len(train_prompts),
                "num_val_prompts": len(val_prompts),
                "vlm_model": VLM_MODEL,
            }
        )
        trainer.logger = wandb
        USE_WANDB = True
        print("W&B run started")
    except Exception as e:
        print(f"W&B init failed (non-blocking): {type(e).__name__}: {e}")
        print("Training will continue without W&B logging.")
else:
    print("W&B disabled (no key provided). Metrics saved to metrics_log.json.")

torch.cuda.empty_cache()
start_time = datetime.now()

print("=" * 60)
print("Starting full GRPO training")
print(f"  Start time       : {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Current step     : {trainer.global_step}")
print(f"  Current epoch    : {trainer.current_epoch}")
print("=" * 60)

try:
    trainer.train()
    print("\nTraining complete!")
except KeyboardInterrupt:
    print("\nTraining interrupted by user.")
    trainer.save_checkpoint('crash_checkpoint')
    print(f"Progress saved to: {OUTPUT_DIR}/crash_checkpoint")
except Exception as e:
    print(f"\nTraining failed: {type(e).__name__}: {e}")
    trainer.save_checkpoint('crash_checkpoint')
    print(f"Progress saved to: {OUTPUT_DIR}/crash_checkpoint")
    raise
finally:
    elapsed = datetime.now() - start_time
    elapsed_h = elapsed.total_seconds() / 3600
    print(f"\nElapsed: {elapsed_h:.2f} hours")
    print(f"Final step: {trainer.global_step}")
    if USE_WANDB:
        try:
            wandb.finish()
        except Exception:
            pass

## Step 12: 保存最终 Checkpoint

In [ ]:
trainer.save_checkpoint("final_checkpoint")

training_history = {
    "global_step"     : int(trainer.global_step),
    "num_epochs"      : NUM_EPOCHS,
    "num_prompts"     : len(train_prompts),
    "lora_rank"       : LORA_RANK,
    "group_size"      : GROUP_SIZE,
    "vlm_model"       : VLM_MODEL,
    "elapsed_hours"   : round(elapsed.total_seconds() / 3600, 2),
}

with open(os.path.join(OUTPUT_DIR, "training_history.json"), "w") as f:
    json.dump(training_history, f, indent=2)

# Save per-step metrics log (for offline plotting without W&B)
metrics_path = os.path.join(OUTPUT_DIR, "metrics_log.json")
with open(metrics_path, "w") as f:
    json.dump(trainer.log_history, f, indent=2)

print(f"Final checkpoint saved: {OUTPUT_DIR}/final_checkpoint")
print(f"Training history saved: {OUTPUT_DIR}/training_history.json")
print(f"Metrics log saved    : {metrics_path}  ({len(trainer.log_history)} entries)")
print(json.dumps(training_history, indent=2))

## Step 12b: 训练曲线可视化（离线，无需 W&B）

从 `trainer.log_history` 或 `metrics_log.json` 绘制 reward / ratio / clip_fraction 曲线。
即使 W&B 不可用，也能看到完整训练轨迹。

In [ ]:
import matplotlib.pyplot as plt
import json, os

# Load from memory or file
log_data = getattr(trainer, 'log_history', None)
if not log_data:
    metrics_path = os.path.join(OUTPUT_DIR, 'metrics_log.json')
    if os.path.exists(metrics_path):
        with open(metrics_path) as f:
            log_data = json.load(f)
        print(f'Loaded {len(log_data)} entries from {metrics_path}')
    else:
        print('No metrics data found. Run training first.')
        log_data = []

if log_data:
    # Separate train vs eval entries
    train_entries = [e for e in log_data if 'train/loss' in e or 'train/reward_mean' in e]
    eval_entries  = [e for e in log_data if 'eval/avg_reward' in e]

    def extract(entries, key):
        steps  = [e['step'] for e in entries if key in e]
        values = [e[key]    for e in entries if key in e]
        return steps, values

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # 1. Reward mean
    ax = axes[0, 0]
    s, v = extract(train_entries, 'train/reward_mean')
    ax.plot(s, v, label='train reward', alpha=0.8)
    se, ve = extract(eval_entries, 'eval/avg_reward')
    if se:
        ax.plot(se, ve, 'ro-', label='eval reward', markersize=5)
    ax.set_title('Reward Mean'); ax.set_xlabel('Step'); ax.legend()

    # 2. Component rewards
    ax = axes[0, 1]
    for key, label in [('train/reward_clip_mean', 'CLIP'), ('train/reward_vlm_mean', 'VLM')]:
        s, v = extract(train_entries, key)
        if s:
            ax.plot(s, v, label=label, alpha=0.8)
    ax.set_title('Component Rewards'); ax.set_xlabel('Step'); ax.legend()

    # 3. Loss
    ax = axes[0, 2]
    s, v = extract(train_entries, 'train/loss')
    ax.plot(s, v, alpha=0.8)
    ax.set_title('Training Loss'); ax.set_xlabel('Step')

    # 4. Ratio mean
    ax = axes[1, 0]
    s, v = extract(train_entries, 'train/ratio_mean')
    if s:
        ax.plot(s, v, alpha=0.8)
        ax.axhline(y=1.0, color='r', linestyle='--', alpha=0.5)
    ax.set_title('Policy Ratio Mean'); ax.set_xlabel('Step')

    # 5. Clip fraction
    ax = axes[1, 1]
    s, v = extract(train_entries, 'train/clip_fraction')
    if s:
        ax.plot(s, v, alpha=0.8)
    ax.set_title('Clip Fraction'); ax.set_xlabel('Step')

    # 6. KL divergence
    ax = axes[1, 2]
    s, v = extract(train_entries, 'train/kl_div')
    if s:
        ax.plot(s, v, alpha=0.8)
    ax.set_title('KL Divergence'); ax.set_xlabel('Step')

    plt.suptitle(f'Training Curves — {trainer.global_step} steps', fontsize=14)
    plt.tight_layout()
    curves_path = os.path.join(OUTPUT_DIR, 'training_curves.png')
    plt.savefig(curves_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Curves saved: {curves_path}')

## Step 13: Before vs After 对比可视化

从每个核心类别各取一个代表性 prompt，展示训练前后对比。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# 覆盖所有核心类别的代表性 prompts
eval_prompts = [
    # color_binding
    "a red apple and a green pear on a table",
    # spatial_relations
    "a cat sitting on top of a box",
    # counting
    "three red apples on a plate",
    # attribute_binding
    "a large red apple and a small green apple",
    # shape_binding
    "a round red ball and a square blue box",
    # texture_binding
    "a rough stone wall next to a smooth glass window",
    # non_spatial
    "a burning candle beside an unlit candle",
    # complex
    "a red apple on a blue plate next to a green cup on a wooden table",
]

print("Generating after-training samples...")
after_images  = []
after_rewards = []

for prompt in eval_prompts:
    torch.cuda.empty_cache()
    img    = generator.generate(prompt=prompt)[0]
    reward = reward_model.compute_reward([img], [prompt]).rewards.item()
    after_images.append(img)
    after_rewards.append(reward)
    print(f"  [{reward:.3f}] {prompt[:50]}")

# Plot
n = len(eval_prompts)
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

categories = [
    "color", "spatial", "counting", "attribute",
    "shape", "texture", "non-spatial", "complex"
]

for i, (prompt, img, reward, cat) in enumerate(zip(
    eval_prompts, after_images, after_rewards, categories
)):
    axes[i].imshow(img)
    axes[i].set_title(
        f"[{cat}] reward={reward:.3f}\n{prompt[:45]}{'...' if len(prompt)>45 else ''}",
        fontsize=8, pad=4
    )
    axes[i].axis('off')

plt.suptitle(
    f"After Full Training ({trainer.global_step} steps, {NUM_EPOCHS} epochs) — "
    f"Avg reward: {sum(after_rewards)/len(after_rewards):.3f}",
    fontsize=13
)
plt.tight_layout()

save_path = os.path.join(OUTPUT_DIR, "after_training_grid.png")
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"\nGrid saved: {save_path}")
print(f"Average reward: {sum(after_rewards)/len(after_rewards):.4f}")

## Step 14: 下载结果

如果使用了 Google Drive，文件已自动保存。
否则运行下方 cell 下载到本地。

In [ ]:
if not USE_GOOGLE_DRIVE:
    !zip -r results_full.zip {OUTPUT_DIR}/
    from google.colab import files
    files.download('results_full.zip')
    print("Results downloaded.")
else:
    print(f"Results already synced to Google Drive:")
    print(f"  {OUTPUT_DIR}/final_checkpoint/")
    print(f"  {OUTPUT_DIR}/training_history.json")
    print(f"  {OUTPUT_DIR}/after_training_grid.png")
    !ls -lh {OUTPUT_DIR}/

---

## 训练完成后的下一步

| 任务 | 负责人 | 说明 |
|------|--------|------|
| Benchmark 评测 | B | T2I-CompBench, GenEval-2 |
| Benchmark 评测 | D | TIFA, GenAI-Bench |
| Before/After 对比分析 | C | 使用本 notebook Step 13 的输出 |
| Error Taxonomy | C | 参考生成结果按类别分析 |
| Ablation 实验 | B | 修改 LORA_RANK / GROUP_SIZE / kl_coef 重跑 |

**Ablation checkpoint 命名建议**（修改 Step 0 中的 OUTPUT_DIR）：
```
grpo_full_lora_r8      # LoRA r=8
grpo_full_lora_r32     # LoRA r=32
grpo_full_clip_only    # 仅 CLIP reward（去掉 VLM）
grpo_full_vlm_only     # 仅 VLM reward（去掉 CLIP）
grpo_full_5ep          # 5 epochs
```